In [2]:
import pickle
import numpy as np
import scipy as sp
from matplotlib import pyplot as plt

import morph_analyses as m

%matplotlib inline


%load_ext autoreload
%autoreload 2

In [2]:

with open(m.params.data_dir / "morph_hists.pkl", 'rb') as file:
    morph_dict = pickle.load(file)
    
rare_dists = m.empirical_priors.prior_post(morph_dict['rare'])
freq_dists = m.empirical_priors.prior_post(morph_dict['frequent'])

In [ ]:
sess = 

In [ ]:
H, H_prior, dkl, dkl_prior = m.similarity_fraction.single_sess_reconstruction(, morph_dict=morph_dict)

In [ ]:
sigma_likelihood=.3

def plot_sfreg(sfreg,cmap):
    cmap = plt.cm.get_cmap(cmap)
    low,high = ut.inv_wallmorphx(-.3),ut.inv_wallmorphx(1.3)
    x = np.linspace(-.3,1.3,num=1000)[np.newaxis,:]

    f,ax = plt.subplots(2,3,figsize=[30,20])

    rare_logpost = np.log(rare_post.T)
    # rare_logpost_spline()
    ax[0,0].imshow(rare_logpost,vmin=np.percentile(rare_logpost,20),extent =[low,high,high,low],cmap='RdPu')
    ax[0,1].imshow(rare_post.T,vmin=np.percentile(rare_post,5),extent =[low,high,high,low],cmap='RdPu')
    ax[0,2].plot(ut.inv_wallmorphx(x.ravel()),rare_prior.ravel(),color='brown',linewidth=5)
    ax[0,2].plot(ut.inv_wallmorphx(x.ravel()),freq_prior.ravel(),color='blue',linewidth=5)

    freq_logpost = np.log(freq_post.T)
    ax[1,0].imshow(freq_logpost,vmin=np.percentile(freq_logpost,20),extent =[low,high,high,low],cmap='RdPu')
    ax[1,1].imshow(freq_post.T,vmin=np.percentile(freq_post,5),extent =[low,high,high,low],cmap='RdPu')
    ax[1,2].plot(ut.inv_wallmorphx(x.ravel()),freq_prior.ravel(),color='blue',linewidth=5)
    
    kllist = []
    trialn = 0
    for i, (mouse, res) in enumerate(sfreg.items()):
#         print(res['wallmorph'])
        try:
            _wallmorph = np.concatenate(res['wallmorph'])
            rare_yhat = np.concatenate(res['rare_yhat'])
            freq_yhat = np.concatenate(res['freq_yhat'])
        except:
            _wallmorph = res['wallmorph']
            rare_yhat = res['rare_yhat']
            freq_yhat = res['freq_yhat']
#         print(_wallmorph,rare_yhat,freq_yhat)
        ax[0,0].scatter(ut.inv_wallmorphx(_wallmorph),ut.inv_wallmorphx(rare_yhat),color=cmap(np.float(i/len(sfreg.keys()))),alpha=.5)
        ax[0,1].scatter(ut.inv_wallmorphx(_wallmorph),ut.inv_wallmorphx(rare_yhat),color=cmap(np.float(i/len(sfreg.keys()))),alpha=.5)
        ax[1,0].scatter(ut.inv_wallmorphx(_wallmorph),ut.inv_wallmorphx(freq_yhat),color=cmap(np.float(i/len(sfreg.keys()))),alpha=.5)
        ax[1,1].scatter(ut.inv_wallmorphx(_wallmorph),ut.inv_wallmorphx(freq_yhat),color=cmap(np.float(i/len(sfreg.keys()))),alpha=.5)

        _x = x.ravel()
        wmsm = u.gaussian(_wallmorph[:,np.newaxis,np.newaxis],.1,_x[np.newaxis,:,np.newaxis])
        rare_yhsm = u.gaussian(rare_yhat[:,np.newaxis,np.newaxis],.1,_x[np.newaxis,np.newaxis,:])
        rare_H = np.sum(wmsm*rare_yhsm,axis=0)
        Z = rare_H.sum(axis=1)
        rare_H/=rare_H.sum(axis=1,keepdims=True)
        rare_H_prior =rare_H.sum(axis=0)/(u.gaussian(x,sigma_likelihood,_x[:,np.newaxis])/Z[:,np.newaxis]).sum(axis=1)
        rare_H_prior /=rare_H_prior.sum()

        freq_yhsm = u.gaussian(freq_yhat[:,np.newaxis,np.newaxis],.1,_x[np.newaxis,np.newaxis,:])
        freq_H = np.sum(wmsm*freq_yhsm,axis=0)
        Z = freq_H.sum(axis=1)
        freq_H/=freq_H.sum(axis=1,keepdims=True)
        freq_H_prior =freq_H.sum(axis=0)/(u.gaussian(x,sigma_likelihood,_x[:,np.newaxis])/Z[:,np.newaxis]).sum(axis=1)
        freq_H_prior /=freq_H_prior.sum()

        ax[0,2].plot(ut.inv_wallmorphx(x.ravel()),rare_H_prior,color=cmap(np.float(i/len(rare_mice))))
        ax[1,2].plot(ut.inv_wallmorphx(x.ravel()),freq_H_prior,color=cmap(np.float(i/len(rare_mice))))
        
        
        rarekl,freqkl = sp.stats.entropy(rare_prior.ravel(),rare_H_prior,base=2),sp.stats.entropy(freq_prior.ravel(),freq_H_prior,base=2)
        print(mouse,rarekl-freqkl)
        kllist.append(rarekl-freqkl)
        trialn+=_wallmorph.shape[0]
#     #     print(np.median(rare_entr-freq_entr))

    print('trial N', trialn)

    # xx = np.linspace(-.1,1.1,num=1000)
    mask = x<np.inf
    xf = ut.inv_wallmorphx(x)
    mask[(xf<ut.xfreq(-.1)) | ((xf>ut.xfreq(.1))&(xf<ut.xfreq(.15))) | ((xf>ut.xfreq(.35))&(xf<ut.xfreq(.4))) | ((xf>ut.xfreq(.6))&(xf<ut.xfreq(.65))) |((xf>ut.xfreq(.85))&(xf<ut.xfreq(.9))) |(xf>ut.xfreq(1.1))]=False
    mask = mask.ravel()
    # mask[]
    rare_ytarget = ut.inv_wallmorphx(x[0,np.argmax(rare_prior*u.gaussian(xx[:,np.newaxis],sigma_likelihood,x),axis=1)].ravel())
    ax[0,0].scatter(ut.inv_wallmorphx(xx[mask]),rare_ytarget[mask],color='brown',zorder=100)
    ax[0,1].scatter(ut.inv_wallmorphx(xx[mask]),rare_ytarget[mask],color='brown',zorder=100)

    freq_ytarget = ut.inv_wallmorphx(x[0,np.argmax(freq_prior*u.gaussian(xx[:,np.newaxis],sigma_likelihood,x),axis=1)].ravel())
    ax[1,0].scatter(ut.inv_wallmorphx(xx[mask]),freq_ytarget[mask],color='blue',zorder=100)
    ax[1,1].scatter(ut.inv_wallmorphx(xx[mask]),freq_ytarget[mask],color='blue',zorder=100)


    ax[0,0].set_xlabel("$\hat{f}_h$",fontsize=30)
    ax[0,0].set_ylabel("$f_h$",fontsize=30)
    ax[0,0].set_title("$Ideal \  Rare \ log(P(f_h|\hat{f}_h))$",fontsize=30)
    ax[0,0].set_xlim([ut.xfreq(-.11),ut.xfreq(1.11)])
    ax[0,0].set_ylim([high,low])

    ax[0,1].set_xlabel("$\hat{f}_h$",fontsize=30)
    ax[0,1].set_ylabel("$f_h$",fontsize=30)
    ax[0,1].set_title("$Ideal \  Rare \ P(f_h|\hat{f}_h)$",fontsize=30)
    ax[0,1].set_xlim([ut.xfreq(-.11),ut.xfreq(1.11)])
    ax[0,1].set_ylim([high,low])

    ax[0,2].set_xlabel("$f_h$",fontsize=30)
    ax[0,2].set_ylabel("$P(f_h)$",fontsize=30)
    ax[0,2].set_title("Reconstructed Prior vs Ideal Rare Prior",fontsize=30)
#     ax[0,2].set_xlim([-.11,1.11])

    ax[1,0].set_xlabel("$\hat{f}_h$",fontsize=30)
    ax[1,0].set_ylabel("$f_h$",fontsize=30)
    ax[1,0].set_title("$Ideal \ Freq. \ log(P(f_h|\hat{f}_h))$",fontsize=30)
    ax[1,0].set_xlim([ut.xfreq(-.11),ut.xfreq(1.11)])
    ax[1,0].set_ylim([high,low])
#
    ax[1,1].set_xlabel("$\hat{f}_h$",fontsize=30)
    ax[1,1].set_ylabel("$f_h$",fontsize=30)
    ax[1,1].set_title("$Ideal \ Freq. \ P(f_h|\hat{f}_h)$",fontsize=30)
    ax[1,1].set_xlim([ut.xfreq(-.11),ut.xfreq(1.11)])
    ax[1,1].set_ylim([high,low])

    ax[1,2].set_xlabel("$f_h$",fontsize=30)
    ax[1,2].set_ylabel("$P(f_h)$",fontsize=30)
    ax[1,2].set_title("Reconstructed Prior vs Ideal Freq. Prior",fontsize=30)
    return f,ax, kllist



In [ ]:
mask = x>np.inf
mask[(ut.inv_wallmorphx(x)>ut.xfreq(-.1))&(ut.inv_wallmorphx(x)<ut.xfreq(.1)) ]=True
mask=mask.ravel()
rare_ytarget = x[0,np.argmax(rare_prior*gaussian(xx[:,np.newaxis],sigma_likelihood,x),axis=1)].ravel()
freq_ytarget = x[0,np.argmax(freq_prior*gaussian(xx[:,np.newaxis],sigma_likelihood,x),axis=1)].ravel()
print(rare_ytarget[mask].mean(),freq_ytarget[mask].mean(),.5*(rare_ytarget[mask].mean()+freq_ytarget[mask].mean()))
print(ut.inv_wallmorphx(rare_ytarget[mask].mean()),ut.inv_wallmorphx(freq_ytarget[mask].mean()),ut.inv_wallmorphx(.5*(rare_ytarget[mask].mean()+freq_ytarget[mask].mean())))
mask = x>np.inf
mask[(ut.inv_wallmorphx(x)>ut.xfreq(.9))&(ut.inv_wallmorphx(x)<ut.xfreq(1.1)) ]=True
mask=mask.ravel()
print(rare_ytarget[mask].mean(),freq_ytarget[mask].mean(),.5*(rare_ytarget[mask].mean()+freq_ytarget[mask].mean()))
print(ut.inv_wallmorphx(rare_ytarget[mask].mean()),ut.inv_wallmorphx(freq_ytarget[mask].mean()),ut.inv_wallmorphx(.5*(rare_ytarget[mask].mean()+freq_ytarget[mask].mean())))


In [ ]:
f,ax,rare_kl = plot_sfreg(rare_sfreg,'copper')

In [ ]:
f,ax,freq_kl = plot_sfreg(freq_sfreg,'cividis')

In [ ]:
f,ax = plt.subplots()
ax.scatter(-.008+np.ones([6,])+.003*np.random.rand(6),np.array(rare_kl),c = plt.cm.copper(np.arange(0,6)/6.),s=200)
print(np.mean(rare_kl),sp.stats.sem(rare_kl))
ax.scatter(np.ones([6,])+.003*np.random.rand(6),np.array(freq_kl),c = plt.cm.cividis(np.arange(0,6)/6.),s=200)
print(np.mean(freq_kl),sp.stats.sem(freq_kl))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.plot([.99,1.01],[0,0],color='red',linewidth=5,zorder=-1)

In [ ]:

def single_session_kldiv(sfreg,saveplots=False,group='rare',fs = False):
    kl_divs={}
    avgH = []
    for i, (mouse, res) in enumerate(sfreg.items()):
        if isinstance(res['wallmorph'],list):
            _wallmorph = np.concatenate(res['wallmorph'])
            _yhat = np.concatenate(res['rare_yhat'])
            nsess = len(res['wallmorph'])
        else:
            _wallmorph = res['wallmorph']
            _yhat = res['rare_yhat']
            nsess = 1
        kl_divs[mouse]=[]
        print(mouse)
        _x = x.ravel()
        _xmask = (_x>=-.1) & (_x<=1.1)
        HH = 0
        
        
        for sess in range(nsess):
            if isinstance(res['wallmorph'],list):
                _wm,_yh = res['wallmorph'][sess],res['rare_yhat'][sess]
            else:
                _wm,_yh = _wallmorph,_yhat
            wmsm = u.gaussian(_wm[:,np.newaxis,np.newaxis],.1,_x[np.newaxis,:,np.newaxis])
            yhsm = u.gaussian(_yh[:,np.newaxis,np.newaxis],.1,_x[np.newaxis,np.newaxis,:])
            H = np.sum(wmsm*yhsm,axis=0)
            Z=H.sum(axis=1)
            H/=H.sum(axis=1,keepdims=True)

            rare_kl,freq_kl = [],[]
            for row in range(H.shape[1]):
                if (_x[row]>=.1) and (_x[row]<=1.1):
                    if fs:
                        rare_kl.append(sp.stats.entropy(rare_fs_post[row,:],H[row,:],base=2))
                        freq_kl.append(sp.stats.entropy(freq_fs_post[row,:],H[row,:],base=2))
                    else:
                        rare_kl.append(sp.stats.entropy(rare_post[row,:],H[row,:],base=2))
                        freq_kl.append(sp.stats.entropy(freq_post[row,:],H[row,:],base=2))
            rare_kl,freq_kl = np.array(rare_kl),np.array(freq_kl)
            kl_divs[mouse].append([rare_kl.mean(),freq_kl.mean()])
        
        
            HH+=H
            f,ax = plt.subplots(1,4,figsize=[20,5])
            ax[0].imshow(H[_xmask,:].T,cmap='RdPu',extent=[ut.inv_wallmorphx(-.1),ut.inv_wallmorphx(1.1),ut.inv_wallmorphx(1.3),ut.inv_wallmorphx(-.3)])
#             ax[0].scatter(ut.inv_wallmorphx(_wm),ut.inv_wallmorphx(_yh),color='blue')
            ax[0].set_xlabel("$\hat{S}$")
            ax[0].set_ylabel("$S$")
            ax[0].set_title("$Q$ mouse %s sess %i diff KL %f" % (mouse, sess,(np.array(rare_kl).mean()-np.array(freq_kl).mean())))
            ax[1].imshow(rare_post[_xmask,:].T,cmap='RdPu',extent=[ut.inv_wallmorphx(-.1),ut.inv_wallmorphx(1.1),ut.inv_wallmorphx(1.3),ut.inv_wallmorphx(-.3)])
            ax[1].set_xlabel("$\hat{S}$")
            ax[1].set_ylabel("$S$")
            ax[1].set_title("Ideal Rare Posterior")
            ax[2].imshow(freq_post[_xmask,:].T,cmap='RdPu',extent=[ut.inv_wallmorphx(-.1),ut.inv_wallmorphx(1.1),ut.inv_wallmorphx(1.3),ut.inv_wallmorphx(-.3)])
            ax[2].set_xlabel("$\hat{S}$")
            ax[2].set_ylabel("$S$")
            ax[2].set_title("Ideal Freq Posterior")
            H_prior = H[_xmask,:].sum(axis=0)/(u.gaussian(x,sigma_likelihood,_x[:,np.newaxis])/Z[:,np.newaxis]).sum(axis=1)
            H_prior /=H_prior.sum()
            ax[3].plot(ut.inv_wallmorphx(_x),H_prior)
            ax[3].plot(ut.inv_wallmorphx(_x),rare_prior.ravel(),color='brown')
            ax[3].plot(ut.inv_wallmorphx(_x),freq_prior.ravel(),color='blue')
            ax[3].set_title("%f" % (sp.stats.entropy(rare_prior.ravel(),H_prior.ravel(),base=2)-sp.stats.entropy(freq_prior.ravel(),H_prior.ravel(),base=2)))
            if saveplots:
                if fs:
                    
                    f.savefig(os.path.join("D:\\Morph_Results\\figures\\KLDivs\\","%s_fs.pdf" % mouse),format="pdf")
                else:
                    f.savefig(os.path.join("D:\\Morph_Results\\figures\\KLDivs\\","%s_sess%i.pdf" % (mouse,sess)),format="pdf")
        
        avgH.append(HH/(sess+1))
       
    avgH = np.nansum(np.array(avgH),axis=0)
    print(avgH.shape)
    avgH/=avgH.sum(axis=1,keepdims=True)
    f,ax = plt.subplots()
#     ax.imshow(avgH.T)
    ax.imshow(avgH.T,cmap='RdPu',extent=[ut.inv_wallmorphx(-.1),ut.inv_wallmorphx(1.1),ut.inv_wallmorphx(1.3),ut.inv_wallmorphx(-.3)])
    
    rare_kl,freq_kl = [],[]
    for row in range(avgH.shape[1]):
        if (_x[row]>=.1) and (_x[row]<=1.1):
            rare_kl.append(sp.stats.entropy(rare_post[row,:],avgH[row,:],base=2))
            freq_kl.append(sp.stats.entropy(freq_post[row,:],avgH[row,:],base=2))
    rare_kl,freq_kl = np.array(rare_kl),np.array(freq_kl)
    
    ax.set_title("diff KL %f" % (np.array(rare_kl).mean()-np.array(freq_kl).mean()))
    
    
    return kl_divs


In [ ]:
rare_kldivs = single_session_kldiv(rare_sfreg,saveplots=False,group='rare')

In [ ]:
freq_kldivs = single_session_kldiv(freq_sfreg,saveplots=False,group='freq')

In [ ]:
f,ax = plt.subplots(figsize=[8,8])
rkl = []
for i,(mouse,kl) in enumerate(rare_kldivs.items()):
    kl = np.array(kl)
    diff = kl[:,0]-kl[:,1]
    rkl.append(diff.mean())
    ax.scatter(i+.25*np.random.rand(diff.shape[0]),diff,color=plt.cm.copper(i/6.),s=200)
print(np.mean(rkl),sp.stats.sem(rkl))
fkl = []
for j, (mouse,kl) in enumerate(freq_kldivs.items()):
    kl = np.array(kl)
    diff = kl[:,0]-kl[:,1]
    fkl.append(diff.mean())
    ax.scatter(i+1+j+.25*np.random.rand(diff.shape[0]),diff,color=plt.cm.cividis(j/6.),s=200)
print(np.mean(fkl),sp.stats.sem(fkl))
ax.set_xticks(np.arange(j+1+i+1))
ax.set_xticklabels(['R1','R2','R3','R4','R5','R6','F1','F2','F3','F4','F5','F6'])
ax.plot([-1,12],[0,0],color='red',linewidth=5,zorder=-1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel("$  D_{KL}(  P_{rare}  ||  Q  ) - D_{KL} (  P_{freq}  ||  Q  ) $",fontsize=15)
# f.savefig("D:\\Morph_Results\\figures\\KLDivs\\Single_Session_Diffs.pdf",format="pdf")

In [ ]:
f,ax = plt.subplots(1,2,figsize=[10,5])
rfskl = []
for i,(mouse,kl) in enumerate(rare_kldivs.items()):
    kl = np.array(kl)
    diff = kl[:,0]-kl[:,1]
    if mouse != '4343706':
        diff0 = rare_fs_kldivs[mouse][0][0]-rare_fs_kldivs[mouse][0][1]
        rfskl.append(diff0)
        ax[0].plot(np.append(3,np.arange(8,8+diff.shape[0])), np.append(diff0,diff),color=plt.cm.copper(i/6.),linewidth=5)
    else:
        ax[0].plot(np.arange(8,8+diff.shape[0]),diff,color=plt.cm.copper(1.),linewidth=5)
ax[0].set_ylim([-3,1.5])
ax[0].set_xticks([3,8,12])
ax[0].spines['top'].set_visible(False)
ax[0].spines['right'].set_visible(False)
ax[0].plot([3,13],[0,0],color='red',linewidth=5,zorder=-1)
print(np.mean(rfskl),sp.stats.sem(rfskl))

ffskl=[]
for i,(mouse,kl) in enumerate(freq_kldivs.items()):
    kl = np.array(kl)
    diff = kl[:,0]-kl[:,1]
    
    diff0 = freq_fs_kldivs[mouse][0][0]-freq_fs_kldivs[mouse][0][1]
    ffskl.append(diff0)
    ax[1].plot(np.append(3,np.arange(8,8+diff.shape[0])), np.append(diff0,diff),color=plt.cm.cividis(i/6.),linewidth=5)
print(np.mean(ffskl),sp.stats.sem(ffskl))
ax[1].set_ylim([-3,1.5])
ax[1].set_xticks([3,8,12])
ax[1].spines['top'].set_visible(False)
ax[1].spines['right'].set_visible(False)
ax[1].plot([3,13],[0,0],color='red',linewidth=5,zorder=-1)
# f.savefig("D:\\Morph_Results\\figures\\KLDivs\\Single_Session_Diffs_V_Time.pdf",format="pdf")

In [ ]:
fd_sfreg = {mouse:single_mouse_sf_regression(mouse,dff,first_ind=2) for mouse,fs in zip(fd_mice,fd_fs)}

In [ ]:
fd_kldivs = single_session_kldiv(fd_sfreg,saveplots=True,group="FD",fs=False)

In [ ]:
f,ax = plt.subplots(figsize=[8,8])
for i,(mouse,kl) in enumerate(rare_kldivs.items()):
    kl = np.array(kl)
    diff = kl[:,0]-kl[:,1]
    ax.scatter(i+.25*np.random.rand(diff.shape[0]),diff,color=plt.cm.copper(i/6.),s=200)
for j, (mouse,kl) in enumerate(freq_kldivs.items()):
    kl = np.array(kl)
    diff = kl[:,0]-kl[:,1]
    ax.scatter(i+1+j+.25*np.random.rand(diff.shape[0]),diff,color=plt.cm.cividis(j/6.),s=200)
    
    
for k, (mouse,kl ) in enumerate(fd_kldivs.items()):
    kl = np.array(kl)
    diff = kl[:,0]-kl[:,1]
    ax.scatter(i+1+j+1+k+.25*np.random.rand(diff.shape[0]),diff,color=plt.cm.viridis(k/4.),s=200)

ax.set_xticks(np.arange(j+1+i+k+1))
ax.set_xticklabels(['R1','R2','R3','R4','R5','R6','F1','F2','F3','F4','F5','F6','FD1','FD2','FD3','FD4'])
ax.plot([-1,16],[0,0],color='red',linewidth=5,zorder=-1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel("$  D_{KL}(  P_{rare}  ||  Q  ) - D_{KL} (  P_{freq}  ||  Q  ) $",fontsize=15)
f.savefig("D:\\Morph_Results\\figures\\KLDivs\\Single_Session_Diffs_wFD.pdf",format="pdf")